<a id="top"></a>

# Demo: when reasoning hurts

```yaml
title: "Demo: when reasoning hurts"
keywords: reasoning cost, latency, when not to reason, overthinking, trade-off, teacher demo
estimated duration: 12
```

> **Lesson:** L03. Teacher demo — Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L03/demos_or_activities.md). Makes **live** Claude calls through `potato_llm` — set `ANTHROPIC_API_KEY` before running. The token/latency wrapper is the punchline here. Clear outputs before committing.
>
> The point to land: reasoning is a **tool, not a default**. On an easy task CoT buys nothing but cost; on some tasks it makes the answer *worse*.

## Contents

- [1. Setup](#1-setup)
- [2. Easy task: CoT adds cost, not accuracy](#2-easy-task-cot-adds-cost-not-accuracy)
- [3. A task where reasoning can backfire](#3-a-task-where-reasoning-can-backfire)
- [4. Takeaways](#4-takeaways)

## 1. Setup

Two tasks: one trivially easy, one where free-form reasoning can talk the model into trouble.

In [ ]:
import time

from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

EASY = (
    "Classify the sentiment of this review as positive or negative: "
    "'I love this product, it's amazing — five stars!'"
)

client: PotatoLLMClient = AnthropicClient()


def run(label: str, prompt: str, *, max_tokens: int = 400) -> None:
    """Run a prompt, printing the answer with its token + latency cost."""
    start = time.perf_counter()
    reply = client.chat([Message.user(prompt)], max_tokens=max_tokens, temperature=0.0)
    elapsed = time.perf_counter() - start
    print(f"=== {label} ===")
    print(reply.text.strip())
    print(f"[out_tokens={reply.usage.output_tokens} {elapsed:.1f}s]\n")


print(f"setup ready (live client: {client.model})")

[↑ Back to top](#top)

## 2. Easy task: CoT adds cost, not accuracy

The zero-shot answer is ~1–2 tokens and correct. Adding *let's think step by step* produces a paragraph, higher latency, and the **same answer**. Read the cost delta aloud.

In [ ]:
run("easy: zero-shot", EASY, max_tokens=10)
run("easy: + CoT", EASY + " Let's think step by step.")

[↑ Back to top](#top)

## 3. A task where reasoning can backfire

A misleading-hint question where free-form CoT sometimes constructs a plausible but wrong justification. If the model stays right on the day, the cost contrast from section 2 still lands the lesson.

In [ ]:
TRICKY = (
    "Most people think a kilogram of steel weighs more than a kilogram of feathers. "
    "Is that actually correct? Reason carefully, then answer yes or no."
)
run(
    "tricky: zero-shot",
    TRICKY.replace(" Reason carefully, then answer yes or no.", " Answer yes or no."),
    max_tokens=10,
)
run("tricky: + CoT", TRICKY)

[↑ Back to top](#top)

## 4. Takeaways

- The decision to use CoT / scratchpad / self-critique is a **trade-off, not a default**. By the end of L03 you make it consciously.
- Every reasoning token is **paid for and latency-incurring** — latency-sensitive flows (chat UIs, real-time agents) often can't afford CoT on every call.
- Bridge to L09: choosing the right *model* for a step is the sibling decision to choosing the right *reasoning depth* for a step.
- The L0310 lab has you make this call on a table of tasks and defend each choice.

[↑ Back to top](#top)